In [5]:
"""
Training and evaluation loop for the Centrality-Aware Attention U-Net, Vmag task.

DDC VARIANT: centrality supervision is derived from the Ddc INPUT matrix,
not from a DQ target — so this version needs no DQ ground truth at all,
just (X, Vmag) pairs like the plain Vmag pipeline.

Predicts per-bus voltage magnitude (Vmag) from the Ddc dependency matrix,
with an auxiliary FC head trained to predict betweenness/closeness
centrality directly from bottleneck features, supervised against
centrality computed from Ddc itself (row/column sums via CentralityLayer).

Caveat: DC-derived centrality is a proxy for true reactive (DQ) centrality
— see centrality_vmag_loss_ddc.py docstring. This is a weaker but
data-free inductive bias compared to the DQ-supervised variant.

Reports:
  - Vmag-level R²/RMSE/MAE                          (the main task)
  - Centrality-level R²/RMSE/MAE, FC head, Ddc-derived (the auxiliary task)
"""

import time
import csv
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from centrality_vmag_model  import CentralityAwareVmagUNet
from dataset                 import get_dataloaders
from centrality_vmag_loss_ddc import CentralityAwareVmagLoss, compute_vmag_full_metrics


# ---------------------------------------------------------------------------
# Case / dataset selection
# ---------------------------------------------------------------------------

CASE_NUMBER = 14           # e.g. 14 -> case14 dataset folder
NUM_BUSES   = CASE_NUMBER  # number of buses in the selected case

DATASET_ROOT = "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/Vmag_prj/V_MAG_dataset"


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CONFIG = {
    # Data
    "train_path":     f"{DATASET_ROOT}/case{CASE_NUMBER}/dataset_train.npz",
    "val_path":       f"{DATASET_ROOT}/case{CASE_NUMBER}/dataset_val.npz",
    "test_path":      f"{DATASET_ROOT}/case{CASE_NUMBER}/dataset_test.npz",
    "target_key":     "Vmag",

    # Model
    "base_channels":  32,
    "attn_heads":     4,
    "num_buses":      NUM_BUSES,

    # Loss weights
    "loss_alpha":       0.5,   # MSE vs MAE weighting in the Vmag term
    "w_vmag":            1.0,
    "w_head_c":          0.3,

    # Training
    "epochs":         100,
    "batch_size":     64,
    "learning_rate":  1e-3,
    "weight_decay":   1e-4,

    # Scheduler
    "lr_patience":    10,
    "lr_factor":      0.5,

    # Early stopping (tracked on val Vmag RMSE — the main task)
    "early_stop_patience": 20,

    # Output
    "checkpoint_path": f"best_model_centrality_vmag_ddc_unet_case{CASE_NUMBER}.pth",
    "history_path":    f"training_history_centrality_vmag_ddc_unet_case{CASE_NUMBER}.csv",

    # Hardware
    "num_workers":    0,
}


# ---------------------------------------------------------------------------
# Single-epoch helpers
# ---------------------------------------------------------------------------

def train_one_epoch(
    model:      nn.Module,
    loader:     DataLoader,
    criterion:  CentralityAwareVmagLoss,
    optimizer:  torch.optim.Optimizer,
    device:     torch.device,
) -> dict[str, float]:
    model.train()
    total_loss_components = {'loss_total': 0.0, 'loss_vmag': 0.0, 'loss_head_c': 0.0}
    all_outputs = {k: [] for k in ['vmag_pred', 'betweenness_head', 'closeness_head']}
    all_X, all_vmag = [], []

    n_samples = 0
    for X, vmag in loader:
        X, vmag = X.to(device), vmag.to(device)
        bs = X.size(0)

        optimizer.zero_grad()
        outputs = model(X)
        loss, breakdown = criterion(outputs, X, vmag)
        loss.backward()
        optimizer.step()

        for k in total_loss_components:
            total_loss_components[k] += breakdown[k] * bs
        n_samples += bs

        for k in all_outputs:
            all_outputs[k].append(outputs[k].detach().cpu())
        all_X.append(X.detach().cpu())
        all_vmag.append(vmag.detach().cpu())

    for k in total_loss_components:
        total_loss_components[k] /= n_samples

    all_outputs = {k: torch.cat(v) for k, v in all_outputs.items()}
    all_X    = torch.cat(all_X)
    all_vmag = torch.cat(all_vmag)

    metrics = compute_vmag_full_metrics(all_outputs, all_X, all_vmag)
    metrics.update(total_loss_components)
    return metrics


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: CentralityAwareVmagLoss,
    device:    torch.device,
) -> dict[str, float]:
    model.eval()
    total_loss_components = {'loss_total': 0.0, 'loss_vmag': 0.0, 'loss_head_c': 0.0}
    all_outputs = {k: [] for k in ['vmag_pred', 'betweenness_head', 'closeness_head']}
    all_X, all_vmag = [], []

    n_samples = 0
    for X, vmag in loader:
        X, vmag = X.to(device), vmag.to(device)
        bs = X.size(0)

        outputs = model(X)
        loss, breakdown = criterion(outputs, X, vmag)

        for k in total_loss_components:
            total_loss_components[k] += breakdown[k] * bs
        n_samples += bs

        for k in all_outputs:
            all_outputs[k].append(outputs[k].cpu())
        all_X.append(X.cpu())
        all_vmag.append(vmag.cpu())

    for k in total_loss_components:
        total_loss_components[k] /= n_samples

    all_outputs = {k: torch.cat(v) for k, v in all_outputs.items()}
    all_X    = torch.cat(all_X)
    all_vmag = torch.cat(all_vmag)

    metrics = compute_vmag_full_metrics(all_outputs, all_X, all_vmag)
    metrics.update(total_loss_components)
    return metrics


# ---------------------------------------------------------------------------
# Main training loop
# ---------------------------------------------------------------------------

def train(cfg: dict = CONFIG):
    cfg = dict(cfg)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Case: {CASE_NUMBER}  |  num_buses: {cfg['num_buses']}")
    print("Model type: centrality_aware_vmag_unet (Ddc-derived centrality)\n")

    # Data — plain (X, Vmag) pairs, no DQ needed
    train_loader, val_loader, test_loader = get_dataloaders(
        train_path  = cfg["train_path"],
        val_path    = cfg["val_path"],
        test_path   = cfg["test_path"],
        batch_size  = cfg["batch_size"],
        num_workers = cfg["num_workers"],
        target_key  = cfg["target_key"],
    )

    # Model
    model = CentralityAwareVmagUNet(
        base_channels = cfg["base_channels"],
        attn_heads    = cfg["attn_heads"],
        num_buses     = cfg["num_buses"],
    ).to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {total_params:,}\n")

    # Loss, optimizer, scheduler
    criterion = CentralityAwareVmagLoss(
        alpha    = cfg["loss_alpha"],
        w_vmag   = cfg["w_vmag"],
        w_head_c = cfg["w_head_c"],
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = cfg["learning_rate"],
        weight_decay = cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        patience  = cfg["lr_patience"],
        factor    = cfg["lr_factor"],
        #verbose   = True,
    )

    # History CSV
    history_path = Path(cfg["history_path"])
    csv_file = history_path.open('w', newline='')
    fieldnames = [
        'epoch', 'lr',
        'train_loss_total', 'train_loss_vmag', 'train_loss_head_c',
        'train_vmag_r2', 'train_vmag_rmse', 'train_vmag_mae',
        'train_head_r2', 'train_head_rmse', 'train_head_mae',
        'val_loss_total', 'val_loss_vmag', 'val_loss_head_c',
        'val_vmag_r2', 'val_vmag_rmse', 'val_vmag_mae',
        'val_head_r2', 'val_head_rmse', 'val_head_mae',
    ]
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    csv_writer.writeheader()

    best_val_rmse = float('inf')   # tracked on val Vmag RMSE (the main task)
    epochs_no_improve = 0

    header = (
        f"{'Epoch':>6} | {'LR':>8} |"
        f" {'TrLoss':>8} {'TrVmagR2':>9} {'TrVmagRMSE':>11} {'TrHeadR2':>9}"
        f" | {'VlLoss':>8} {'VlVmagR2':>9} {'VlVmagRMSE':>11} {'VlHeadR2':>9}"
    )
    print(header)
    print("-" * len(header))

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_m   = evaluate(model, val_loader, criterion, device)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_m['vmag_rmse'])

        print(
            f"{epoch:>6} | {current_lr:>8.2e} |"
            f" {train_m['loss_total']:>8.4f} {train_m['vmag_r2']:>9.4f}"
            f" {train_m['vmag_rmse']:>11.5f} {train_m['head_r2']:>9.4f}"
            f" | {val_m['loss_total']:>8.4f} {val_m['vmag_r2']:>9.4f}"
            f" {val_m['vmag_rmse']:>11.5f} {val_m['head_r2']:>9.4f}"
            f"  [{time.time()-t0:.1f}s]"
        )

        csv_writer.writerow({
            'epoch': epoch, 'lr': current_lr,
            'train_loss_total':   train_m['loss_total'],
            'train_loss_vmag':    train_m['loss_vmag'],
            'train_loss_head_c':  train_m['loss_head_c'],
            'train_vmag_r2':      train_m['vmag_r2'],
            'train_vmag_rmse':    train_m['vmag_rmse'],
            'train_vmag_mae':     train_m['vmag_mae'],
            'train_head_r2':      train_m['head_r2'],
            'train_head_rmse':    train_m['head_rmse'],
            'train_head_mae':     train_m['head_mae'],
            'val_loss_total':     val_m['loss_total'],
            'val_loss_vmag':      val_m['loss_vmag'],
            'val_loss_head_c':    val_m['loss_head_c'],
            'val_vmag_r2':        val_m['vmag_r2'],
            'val_vmag_rmse':      val_m['vmag_rmse'],
            'val_vmag_mae':       val_m['vmag_mae'],
            'val_head_r2':        val_m['head_r2'],
            'val_head_rmse':      val_m['head_rmse'],
            'val_head_mae':       val_m['head_mae'],
        })
        csv_file.flush()

        if val_m['vmag_rmse'] < best_val_rmse:
            best_val_rmse = val_m['vmag_rmse']
            epochs_no_improve = 0
            torch.save({
                'epoch':             epoch,
                'model_state_dict':  model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_vmag_rmse':     best_val_rmse,
                'val_vmag_r2':       val_m['vmag_r2'],
                'val_head_r2':       val_m['head_r2'],
                'config':            cfg,
            }, cfg["checkpoint_path"])
            print(f"         ↳ New best val Vmag RMSE: {best_val_rmse:.5f}  → saved checkpoint")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg["early_stop_patience"]:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

    csv_file.close()
    print(f"\nTraining complete. Best val Vmag RMSE: {best_val_rmse:.5f}")
    print(f"Checkpoint saved to : {cfg['checkpoint_path']}")
    print(f"History saved to    : {cfg['history_path']}")

    # ------------------------------------------------------------------
    # Final test evaluation with best checkpoint
    # ------------------------------------------------------------------
    print("\n--- Test Evaluation (best checkpoint) ---")
    checkpoint = torch.load(cfg["checkpoint_path"], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    test_m = evaluate(model, test_loader, criterion, device)

    print("Vmag-level (per-bus voltage magnitude):")
    print(f"  R²  : {test_m['vmag_r2']:.4f}")
    print(f"  RMSE: {test_m['vmag_rmse']:.6f}")
    print(f"  MAE : {test_m['vmag_mae']:.6f}")
    ##print("\nCentrality-level (betweenness + closeness, FC head, Ddc-derived, auxiliary):")
    #print(f"  R²  : {test_m['head_r2']:.4f}")
    #print(f"  RMSE: {test_m['head_rmse']:.6f}")
    #print(f"  MAE : {test_m['head_mae']:.6f}")

    return model, test_m


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    train(CONFIG)


Device: cuda
Case: 14  |  num_buses: 14
Model type: centrality_aware_vmag_unet (Ddc-derived centrality)

Train : DependencyMatrixDataset(n=7734, target_key='Vmag', X=(7734, 1, 14, 14), y=(7734, 14))
Val   : DependencyMatrixDataset(n=1657, target_key='Vmag', X=(1657, 1, 14, 14), y=(1657, 14))
Test  : DependencyMatrixDataset(n=1659, target_key='Vmag', X=(1659, 1, 14, 14), y=(1659, 14))
Model parameters: 2,744,575

 Epoch |       LR |   TrLoss  TrVmagR2  TrVmagRMSE  TrHeadR2 |   VlLoss  VlVmagR2  VlVmagRMSE  VlHeadR2
-------------------------------------------------------------------------------------------------------
     1 | 1.00e-03 |   1.0798 -510.0962     0.47647    0.2499 |   0.1045   -0.2681     0.02374    0.9176  [2.9s]
         ↳ New best val Vmag RMSE: 0.02374  → saved checkpoint
     2 | 1.00e-03 |   0.0929    0.8414     0.00839    0.9209 |   0.0309    0.5210     0.01459    0.9785  [2.3s]
         ↳ New best val Vmag RMSE: 0.01459  → saved checkpoint
     3 | 1.00e-03 |   0.05

Here's what each printed value means:

Setup: this block loads the checkpoint with the lowest validation Vmag RMSE seen during training, then runs it once on the held-out test set (test_loader), which the model never saw during training or model selection.

Vmag-level (per-bus voltage magnitude) — how well the model predicts the actual target:

R²: fraction of variance in the true Vmag values explained by the predictions, computed over all buses and all test samples flattened together. 1.0 = perfect, 0.0 = no better than predicting the mean, negative = worse than predicting the mean.
RMSE: root-mean-squared error between predicted and true Vmag, in the same units as Vmag (typically per-unit voltage, e.g. ~0.95–1.05 range). Penalizes large errors more heavily than MAE.
MAE: mean absolute error — the average magnitude of the error, same units as Vmag, more robust to outliers than RMSE.

Centrality-level (betweenness + closeness, FC head, Ddc-derived, auxiliary) — how well the auxiliary branch does at its side task, not the main deliverable:

These are the same three metrics (R², RMSE, MAE), but computed on the auxiliary CentralityHead's predicted betweenness+closeness vectors (concatenated together) against betweenness/closeness computed by summing rows/columns of the Ddc input matrix — not the actual electrical centrality, since this is the Ddc-derived variant.
This tells you whether the auxiliary task is being learned at all (useful for sanity-checking that w_head_c is doing something), but since it's compared against a DC-flow proxy rather than true reactive centrality, don't read too much physical meaning into the absolute numbers — treat it as a training diagnostic more than a result you'd report as ground truth accuracy.

One thing to watch: if head_r2 stays very negative for a long time while vmag_r2 improves normally, that's a sign the auxiliary head isn't learning useful structure (possibly w_head_c too small, or the centrality targets need normalizing) — worth checking against your training curve.